# Single Image Explanation

This notebook demonstrates how to generate saliency maps for a single image using the causal-explainer framework.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np

from causal_explainer import SFL, RelevanceScore, SaliencyMapVisualizer, SFLVisualizer
from causal_explainer.utils import get_device, read_tensor, get_class_name

In [ ]:
# Load model
device = get_device()
print(f'Using device: {device}')

model = models.resnet50(pretrained=True)
model = nn.Sequential(model, nn.Softmax(dim=1))
model = model.eval().to(device)
for p in model.parameters():
    p.requires_grad = False

In [ ]:
# Load image and get prediction
img_path = '../data_demo/catdog.png'
img_tensor = read_tensor(img_path).to(device)

with torch.no_grad():
    output = model(img_tensor)
    top_prob, top_class = torch.topk(output, k=5, dim=1)

for i in range(5):
    print(f'{get_class_name(top_class[0][i].item())}: {top_prob[0][i].item()*100:.2f}%')

target_class = top_class[0][0].item()
print(f'\nTarget class: {target_class} ({get_class_name(target_class)})')

In [ ]:
# Generate mutants
N = 100  # Number of mutants (increase for better quality, e.g. 500-1000)
s = 8    # Mask grid size
p1 = 0.2 # Mask probability

sfl = SFL(model, input_size=(224, 224))
masks, sampled_tensor = sfl.generate_mutants_batch(
    img_path, N, s, p1, target_class
)

In [ ]:
# Compute confidence scores for each mutant
confidence_scores = np.zeros(N)
with torch.no_grad():
    for j in range(N):
        single_image = sampled_tensor[j].unsqueeze(0)
        output = model(single_image)
        confidence_scores[j] = torch.max(output, dim=1).values.item() * 100

In [ ]:
# Calculate SFL relevance scores
rs = RelevanceScore()
pixel_dataset, ochiai, tarantula, zoltar, wong1 = rs.run(
    confidence_scores, sampled_tensor, masks, N
)

In [ ]:
# Visualize saliency maps (all 4 formulas)
viz = SaliencyMapVisualizer(img_path)
viz.visualize_pixel_scores(pixel_dataset, ins='')

In [ ]:
# Visualize a single formula
viz.visualize_pixel_scores(pixel_dataset, ins='ochiai')

In [ ]:
# Inspect individual mutants
pass_fail = ['pass' if i % 2 == 0 else 'fail' for i in range(N)]
mutant_viz = SFLVisualizer()
mutant_viz.interactive_mutant_visualization(sampled_tensor, pass_fail)